# 0003 — E1 engine and walk-forward: acceptance and a first IBrX 100 study

| | |
|---|---|
| **Hypothesis** | (1) The E1 engine reproduces notebook 0002's point-in-time line, so the engine adds nothing to the numbers that the loop by hand did not. (2) An equal-weight portfolio of the IBrX 100, rebuilt in monthly walk-forward folds, is the baseline any strategy on this universe must beat. |
| **Universe** | (1) `Index("IBOV")` per period, as in 0002. (2) `IndexComposition("IBXX")`: the quadrimester's members from its first session, with 252 bars of warmup plus 12 months of history. |
| **Period** | (1) 2003-01 → last session. (2) 2018-01 → last session; folds of 12 months of training and 1 of test. |
| **Result** | Run on 2026-11-19, base to 2026-11-06. (1) 5,912 sessions, largest daily difference 8.9e-16. (2) 107 folds, 2018-01-02 → 2026-11-06; on average 3.8 members a fold left out for short history and 0.4 for stopping. Executed at the next open: CAGR 6.6% vs BOVA11 9.5% (excess −2.9% a year), volatility 26.6% vs 23.9%, Sharpe 0.38 vs 0.50 (risk-free 0), max drawdown −49% vs −47%, beta 1.04, correlation 0.93, information ratio −0.20. Next close: 6.4%. Turnover 4.3× a year: 10 bps per side cost 0.46 point of CAGR, 20 bps 0.92. Worst session 2026-10-26, the day after the election: −21.4% vs BOVA11 −14.5% (MGLU3 −58%, VAMO3 −57% that day, confirmed in the press). |
| **Day-trade** | Section 5 runs the E1 day-trade engine on the same folds: a smoke test, not a strategy. Pending its first run. |
| **Conclusion** | E1 is accepted (B1.22): it reproduces 0002 to rounding, after the first run made it stop holding securities it cannot price. Equal weight on the IBrX 100 is a high-beta, fat-tailed version of the cap-weighted index: since 2018 it lost 2.9 points a year to BOVA11, with more exposure to mid caps hit by rates (2020-03, 2026-10). It is the baseline a strategy on this universe must beat, after costs: at a realistic 10 bps, 6.2% a year. |
| **Drove in `src/`** | E1 trades a security without a bar at its last close instead of keeping its units (B1.12): the first run held ELPL4's old ISIN for six years and booked +345% when it traded again. `Fold.decisions`, `Fold.load_train` and `Fold.load_test`, so that a strategy that fits nothing reads no bars per fold; the provider reads the ledger's jumps once and logs each one once. |

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ibexQuant.analytics import B3_SESSIONS_PER_YEAR, drawdown, performance_summary
from ibexQuant.backtest import (
    DayTradeOrders,
    equal_weight,
    simulate_day_trades,
    simulate_weights,
)
from ibexQuant.backtest.walkforward import IndexComposition, WalkForward, WalkForwardSchedule
from ibexQuant.data import B3Dataset
from ibexQuant.data.universe import Index, Securities

In [ ]:
pd.set_option("display.precision", 4)

ds = B3Dataset.open()
END = ds.calendar.sessions[-1]
print(f"base: {ds.calendar.sessions[0].date()} → {END.date()}")

## 1. Acceptance: E1 against notebook 0002

Notebook 0002 buys the members of each IBOV period in equal weight at the close of the
period's first session and holds them, drifting, until the next one. In E1 that is one row
of weights per period, decided at the close of the session **before** the period starts and
executed at the **next close** (`execution="next_close"`).

The loop below is 0002's own code. It also writes down the weights E1 needs: the members
0002 held, `1 / n` each.

In [ ]:
rebalances = ds.rebalance_dates("IBOV", "2003-01-01", END)
edges = [*rebalances, END]


def notebook_0002() -> tuple[pd.Series, pd.DataFrame]:
    """The point-in-time line of notebook 0002, and the weights E1 needs to repeat it."""
    pieces: list[pd.Series] = []
    rows: dict[pd.Timestamp, pd.Series] = {}
    for entry, exit_ in zip(edges[:-1], edges[1:], strict=True):
        if entry >= exit_:
            continue
        panel = ds.panel(Index("IBOV"), entry, exit_, fields=["close"])
        close = panel.field("close")
        held = close.columns[panel.universe_mask.loc[entry] & close.loc[entry].notna()]
        value = (close[held].ffill() / close.loc[entry, held]).mean(axis=1)
        pieces.append(value.pct_change().iloc[1:])
        rows[ds.calendar.offset(entry, 1)] = pd.Series(1 / len(held), index=held)
    weights = pd.DataFrame(rows).T.fillna(0.0)
    return pd.concat(pieces).rename("notebook_0002"), weights


reference, weights_0002 = notebook_0002()
print(f"{len(weights_0002)} rebalances, {weights_0002.shape[1]} securities")

In [ ]:
bars_0002 = ds.panel(
    Securities(list(weights_0002.columns)), weights_0002.index[0], END, fields=["close"]
)
e1_0002 = simulate_weights(bars_0002, weights_0002, execution="next_close")

compared = pd.concat([reference, e1_0002.returns], axis=1, join="inner")
gap = (compared["notebook_0002"] - compared["portfolio"]).abs()
print(f"{len(compared)} sessions; largest difference {gap.max():.2e} on {gap.idxmax().date()}")
gap[gap > 1e-10]

The two lines must agree to rounding (B1.22): an empty list above. The first run found one
session that did not, 2010-12-21: E1 then kept the units of a security without a bar, and
held a ticker's old ISIN (ELPL4, `BRELPLACNPR6`) for years at a stale price until it traded
again at 300 times that price. E1 now trades a security without a bar at its last close,
as 0002 does (B1.12).

## 2. Walk-forward on the IBrX 100

Folds of 12 months of training and 1 month of test from 2018-01. Each fold trades the
members of the quadrimester's portfolio that have 252 bars plus 12 months of history and
still trade at the decision.

In [ ]:
schedule = WalkForwardSchedule("2018-01", END.strftime("%Y-%m"), train_months=12, test_months=1)
walk = WalkForward(schedule, IndexComposition("IBXX"), warmup=252)
folds = walk.folds(ds)
summary = WalkForward.summary(folds)
first, last = summary["test_start"].iloc[0].date(), summary["test_end"].iloc[-1].date()
print(f"{len(folds)} folds, {first} → {last}")
summary.tail()

In [ ]:
# Where the compositions come from, and who was left out.
print(summary.groupby("source")["period"].nunique().to_string(), "\n")
summary.iloc[:, summary.columns.get_loc("securities") + 1 :].sum().rename("dropped")

Equal weight needs no bars per fold: only who is eligible and when the decisions are
(`fold.decisions`). Weights are decided at every close, so the portfolio is rebalanced back
to equal weight every session. The bars of the simulation are read once.

In [ ]:
pieces = []
for fold in folds:
    eligible = pd.DataFrame(True, index=fold.decisions(ds.calendar), columns=list(fold.securities))
    pieces.append(equal_weight(eligible))
weights = pd.concat(pieces).fillna(0.0)

bars = WalkForward.bars(ds, folds)
runs = {
    execution: simulate_weights(bars, weights, execution=execution)
    for execution in ("next_open", "next_close")
}
result = runs["next_open"]

In [ ]:
benchmark = (
    ds.panel(["BOVA11"], result.returns.index[0], END, fields=["close"])
    .field("close")
    .iloc[:, 0]
    .pct_change()
    .dropna()
    .rename("BOVA11")
)
result.summary(periods_per_year=B3_SESSIONS_PER_YEAR, benchmark=benchmark)

In [ ]:
# Execution at the next open or the next close.
by_execution = pd.concat({name: run.returns for name, run in runs.items()}, axis=1)
performance_summary(by_execution, periods_per_year=B3_SESSIONS_PER_YEAR)

## 3. Where the return came from

`contributions` is each security's profit and loss over the equity of the previous close.
Summed over the sessions, it approximates each security's share of the compounded return.

In [ ]:
history = ds.ticker_history(list(bars.symbols))
ticker = history.sort_values("last_session").groupby("security_id")["ticker"].last()
total = result.contributions.sum().rename(index=ticker).sort_values()
best, worst = total.tail(10)[::-1], total.head(10)
pd.DataFrame(
    {
        "best": best.index,
        "best_contribution": best.to_numpy(),
        "worst": worst.index,
        "worst_contribution": worst.to_numpy(),
    }
)

In [ ]:
# The worst sessions, the three securities that cost most on each, and BOVA11 that day.
for day, value in result.returns.nsmallest(5).items():
    worst = result.contributions.loc[day].rename(index=ticker).nsmallest(3)
    names = ", ".join(f"{name} {share:.2%}" for name, share in worst.items())
    print(f"{day.date()}  {value:7.2%}  BOVA11 {benchmark.get(day, float('nan')):7.2%}  {names}")

A worst session far below BOVA11's, or one security costing several percent of the
portfolio in a day, points to a data problem (an unexplained jump, P13) rather than to the
market.

## 4. Costs

Turnover comes from three sources: daily rebalancing back to equal weight, the monthly
change of eligible members and the quadrimester's portfolio changes. A cost per side in
basis points of the traded notional (B1.13).

In [ ]:
print(f"turnover: {result.turnover.mean() * B3_SESSIONS_PER_YEAR:.2f} times the equity a year")
with_costs = pd.concat(
    {
        f"{bps} bps": simulate_weights(bars, weights, cost_bps=bps).returns
        for bps in (0, 5, 10, 20)
    },
    axis=1,
)
table = performance_summary(with_costs, periods_per_year=B3_SESSIONS_PER_YEAR).metrics
table.loc[
    [("return", "cagr"), ("risk", "annualized_volatility"), ("risk_adjusted", "sharpe_ratio")]
]

## Charts

In [ ]:
common = pd.concat([result.returns, benchmark], axis=1, join="inner")
fig, (ax_growth, ax_drawdown, ax_members) = plt.subplots(
    3, 1, figsize=(12, 10), sharex=True, height_ratios=[2, 1, 1]
)
(1 + common).cumprod().plot(ax=ax_growth, logy=True, title="Growth of 1 (log scale)")
drawdown(common).plot(ax=ax_drawdown, title="Drawdown", legend=False)
(result.weights.drop(columns="cash") > 0).sum(axis=1).plot(ax=ax_members, title="Securities held")
for ax in (ax_growth, ax_drawdown, ax_members):
    ax.grid(alpha=0.3)
fig.tight_layout()

## 5. Day-trade engine: a smoke test

Not a strategy: a check that the day-trade engine runs on real bars (B1.17-B1.21). At each
close, for every eligible member, a buy limit 1% below the close and a sell limit 1% above,
each of `1 / n` of the equity. An order fills only if the price trades strictly through it
(or the opening auction crosses it, at the open); every position is flat at the close.

In [ ]:
ohlc = WalkForward.bars(ds, folds, fields=("open", "high", "low", "close"))
close = ohlc.field("close")
reference = pd.concat(
    [close.loc[fold.decisions(ds.calendar), list(fold.securities)] for fold in folds]
)
quoted = reference.notna()
size = quoted.div(quoted.sum(axis=1), axis=0)
orders = DayTradeOrders(size=size, buy_limit=reference * 0.99, sell_limit=reference * 1.01)
day_trades = {bps: simulate_day_trades(ohlc, orders, cost_bps=bps) for bps in (0, 5, 10)}
day = day_trades[0]
print(f"{len(day.trades)} fills on {day.trades['entry_time'].nunique()} sessions")
day.trades.groupby(["side", "exit_reason"])["return"].describe()

In [ ]:
day.trade_summary()

In [ ]:
by_cost = pd.concat({f"{bps} bps": run.returns for bps, run in day_trades.items()}, axis=1)
performance_summary(by_cost, periods_per_year=B3_SESSIONS_PER_YEAR)

## Known simplifications

- **Adjusted prices, fractional units, no lot or tick** (B1.15): dividends are reinvested in
  each security; raw execution with event cash is E2's (P9, P11).
- **Cash earns nothing** (P14): Sharpe ignores the CDI.
- **An exit is cash at the last close** (B1.12): holders of JBSS3, BRFS3 and AXIA6 received
  other securities, not cash (P10 in E2).
- **Linear costs only**: no borrow fee, no impact, no minimum fee.
- **Day trades on adjusted bars** (B1.20): levels and bars share one scale; no tick or lot,
  and an order of any size fills at its price.
- **2026-05 composition from the 2nd preview** until the definitive file is in the base.